# RAG quick start

Search your own notes from Python.

## What "the index" is

Two different things share that name, which is most of the confusion:

1. **The index — the data.** Your 144 files, chopped into 3055 passages
   ("chunks"), split at your headings. Each chunk is stored alongside ~1000 numbers describing its
   *meaning*. This was built once by `rag index` in a terminal. It sits on disk and does not change
   until you rebuild it.
2. **`index` — the variable.** Just a handle to that data, created in section 1 below.

A search turns your question into numbers the same way, then finds the chunks whose numbers are
closest. That is why an English question can find a German note: it matches meaning, not spelling.

## What each cell costs

**No cell in this notebook ever re-reads or re-embeds your files.** That only happens in a
terminal, via `rag index` / `rag update`. Here, the only thing ever embedded is the short question
you type.

| Tag | What it means |
|---|---|
| `READS INDEX` | reads stored passages from disk — fast |
| `LOADS MODELS` | pulls ~6.4 GB of models into RAM. Slow the **first** time in a kernel session, free every time after |
| `EMBEDS QUERY` | turns your one question into numbers — milliseconds |
| `PURE` | plain Python over data already in memory. No disk, no network, instant |
| `WRITES` | changes something on disk. **Nothing here does.** |

So: the only slow cell is the first search. Everything after it is fast, and nothing you do in this
notebook can damage the index.

## 1. Open the index
### `READS INDEX` · no models · instant

Finds the nearest `.rag/` workspace and opens it, then prints what is stored. Models are loaded
lazily, so nothing heavy happens yet.

This cell also checks you are running on the right kernel. If you are not, it stops here with
instructions instead of failing later with a confusing error.

In [3]:
import sys
from pathlib import Path

# --- kernel guard -----------------------------------------------------------
# This notebook MUST run on the vault's own venv (<vault>/.venv). Selecting any
# other kernel imports rag_toolkit fine (it is pure-python and on sys.path below)
# and then dies much later on `import sentence_transformers`, which is confusing.
# Fail here instead, with the fix.
try:
    import sentence_transformers  # noqa: F401
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Wrong kernel.\n"
        f"  running on : {sys.executable}\n"
        "  expected   : <vault>/.venv/bin/python\n"
        "\n"
        "Pick the kernel named 'my-wiki RAG (.venv)':\n"
        "  VS Code  -> 'Select Kernel' (top right) -> Jupyter Kernel... -> my-wiki RAG (.venv)\n"
        "  Jupyter  -> Kernel -> Change Kernel -> my-wiki RAG (.venv)\n"
        "\n"
        "If it is not listed, register it once:\n"
        "  <vault>/.venv/bin/python -m ipykernel install --user \\\n"
        "      --name my-wiki-rag --display-name 'my-wiki RAG (.venv)'"
    ) from exc

# Put the vendored toolkit on the path. The installer writes a rag_toolkit.pth,
# but macOS flags files under a dot-directory in an iCloud container UF_HIDDEN,
# and CPython 3.13+ skips hidden .pth files — so do not rely on it.
_toolkit = Path.cwd()
while _toolkit != _toolkit.parent and not (_toolkit / ".rag" / "toolkit").is_dir():
    _toolkit = _toolkit.parent
_toolkit = str(_toolkit / ".rag" / "toolkit")
if _toolkit not in sys.path:
    sys.path.insert(0, _toolkit)

from rag_toolkit import Index, to_markdown
from IPython.display import Markdown

index = Index.find()          # or Index.at('/path/to/project')
status = index.status()

print(f"kernel    : {sys.executable}")
print(f"workspace : {status['rag_dir']}")
print(f"indexed   : {status.get('files', 0)} files, {status.get('chunks', 0)} chunks")
print(f"model     : {status['embedding'].get('model')}")
if status.get('warning'):
    print(f"\nWARNING: {status['warning']}")

kernel    : /Users/Khaled.Alabsi/.local/share/rag/my-wiki/venv/bin/python
workspace : /Users/Khaled.Alabsi/Library/Mobile Documents/iCloud~md~obsidian/Documents/my-wiki/.rag
indexed   : 144 files, 3055 chunks
model     : BAAI/bge-m3


## 2. Search
### `LOADS MODELS` (first run only) · `EMBEDS QUERY` · `READS INDEX`

**This is the slow cell — once per kernel session.** Loading the embedding model and the reranker
takes a few seconds. Every later search reuses them from RAM and returns in well under a second.

It embeds *your question* — one short string. It does not touch your notes.

`hits` is a plain list, best match first. `hits[0]` is the top result; `hits[0]['text']` is its
content.

Reading a result line:

```
0.6664  PhD/myt-decomposition.md:65-86 — MYT Decomposition ... > Worked TEP Example
  ^         ^file                 ^lines    ^which heading it sits under
score
```

**Scores only mean something *within one result list*.** 0.67 sitting above 0.19 means the first is
clearly the better match. A whole list down in the thousandths means the answer is not in your
notes — that is a useful answer, not a failure.

In [4]:
hits = index.search('worked example of MYT decomposition on Tennessee Eastman data', k=5)

for hit in hits:
    print(f"{hit['score']:>8.4f}  {hit['citation']}")

The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 45731.38it/s]
/Users/Khaled.Alabsi/Library/Mobile Documents/iCloud~md~obsidian/Documents/my-wiki/.rag/toolkit/rag_toolkit/embed.py:108: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = int(self._model.get_sentence_embedding_dimension() or 0)


  0.6664  PhD/myt-decomposition.md:65-86 — MYT Decomposition (Mason, Young & Tracy, 1995) > Worked TEP Example
  0.1985  PhD/Noise Handling in Statistical and Multivariate Process Monitoring_ A Literature Review.md:3-8 — Noise Handling in Statistical Process Monitoring (SPM) and Multivariate Statistical Process Control (MSPC): A Literature Review > TL;DR
  0.1890  PhD/myt-decomposition.md:87-96 — MYT Decomposition (Mason, Young & Tracy, 1995) > The Core Limitation
  0.1575  PhD/myt-decomposition.md:3-11 — MYT Decomposition (Mason, Young & Tracy, 1995) > Table of Contents
  0.1460  PhD/myt-decomposition.md:16-26 — MYT Decomposition (Mason, Young & Tracy, 1995) > The Formula


## 3. Read the passages
### `PURE` — no search, no models, instant

The same hits as above, showing their actual text instead of just filenames. **No search runs
here** — this only reformats what section 2 already returned.

`max_chars` truncates explicitly, so nothing is silently cut off.

In [5]:
Markdown(to_markdown(hits, max_chars=600))

**1. PhD/myt-decomposition.md:65-86 — MYT Decomposition (Mason, Young & Tracy, 1995) > Worked TEP Example** — score `0.666405` (rerank)

> ## Worked TEP Example
> 
> Baseline statistics from NOC training data, and a faulty sample (e.g. a cooling-water-related fault):
> 
> | Variable | Normal mean $\mu_j$ | Normal std dev $\sigma_j$ | Faulty value $x_j$ | Deviation $d_j = x_j-\mu_j$ | Contribution $d_j^2/\sigma_j^2$ |
> |---|---|---|---|---|---|
> | Reactor Temp | 120.4 | 0.3 | 121.7 | 1.3 | 18.8 |
> | Reactor Pressure | 2705 | 13 | 2718 | 13 | 1.0 |
> | CW Outlet Temp | 94.6 | 1.5 | 98.9 | 4.3 | 8.2 |
> | CW Flow | 41.1 | 2.0 | 46.0 | 4.9 | 6.0 |
> | Reactor Level | 75.0 | 1.8 | 75.3 | 0.3 | 0.03 |
> 
> **Ranking by contribution:** Reactor Temp (18.8) >…

**2. PhD/Noise Handling in Statistical and Multivariate Process Monitoring_ A Literature Review.md:3-8 — Noise Handling in Statistical Process Monitoring (SPM) and Multivariate Statistical Process Control (MSPC): A Literature Review > TL;DR** — score `0.198506` (rerank)

> ## TL;DR
> 
> - **The state of the art splits into two paradigms.** *Noise isolation/characterization* uses latent-variable projection (PCA, PLS, ICA, factor analysis, probabilistic PCA), multiscale wavelet decomposition, and robust statistics to separate a “noise subspace” from a “signal subspace”; *noise reduction/filtering* uses classical smoothing (moving average, median, Savitzky–Golay), model-based filters (Kalman, particle), transform-domain denoising (wavelet, Fourier, EMD/SSA), and modern deep denoising (denoising/variational autoencoders, diffusion models). Most methods still implicitly…

**3. PhD/myt-decomposition.md:87-96 — MYT Decomposition (Mason, Young & Tracy, 1995) > The Core Limitation** — score `0.188965` (rerank)

> ## The Core Limitation
> 
> Sum the contributions above: $18.8 + 1.0 + 8.2 + 6.0 + 0.03 \approx 34$. This does **not** equal the true $T^2$ for the sample, because MYT throws away every cross-covariance term in $\Sigma^{-1}$.
> 
> Concrete illustration with two correlated variables (correlation strength 0.8, meaning they normally move together): if both deviate by the same amount, $d = (2, 2)$:
> - True $T^2 \approx 4.4$ — small, because moving together is *normal* for these two variables.
> - MYT contributions: $4$ and $4$, summing to $8$ — nearly double the true value.
> 
> **Why:** MYT has no concept of "t…

**4. PhD/myt-decomposition.md:3-11 — MYT Decomposition (Mason, Young & Tracy, 1995) > Table of Contents** — score `0.157548` (rerank)

> ## Table of Contents
> - [[#What Problem MYT Solves|What Problem MYT Solves]]
> - [[#The Formula|The Formula]]
> - [[#Why This Formula, Specifically|Why This Formula, Specifically]]
> - [[#Why Divide Instead of Subtract|Why Divide Instead of Subtract]]
> - [[#Worked TEP Example|Worked TEP Example]]
> - [[#The Core Limitation|The Core Limitation]]
> - [[#Summary Table|Summary Table]]

**5. PhD/myt-decomposition.md:16-26 — MYT Decomposition (Mason, Young & Tracy, 1995) > The Formula** — score `0.145963` (rerank)

> ## The Formula
> 
> $$\text{contrib}_j = \frac{d_j^2}{\sigma_j^2}$$
> 
> Where:
> - $j$ = index of the variable (1 through 52 for TEP)
> - $d_j$ = deviation of variable $j$, defined as $d_j = x_j - \mu_j$, where $x_j$ is the variable's current (possibly faulty) value and $\mu_j$ is its normal average value (learned from NOC — normal operating condition — training data)
> - $\sigma_j$ = the normal standard deviation of variable $j$ (how much it naturally fluctuates under normal operation), so $\sigma_j^2$ is its variance
> 
> In plain terms: square the deviation, divide by the variable's own normal variance. Thi…


## 4. Filter
### `EMBEDS QUERY` · `READS INDEX` — fast, models already warm

A second search, restricted to part of the corpus. Filters narrow **before** ranking, so weaker
matches get a chance to surface — that is not the same as searching everything and ignoring the
misses.

> ⚠️ **This overwrites `hits`.** Sections 3, 5, 6 and 7 read that variable, so after running this
> cell they will show *these* filtered results, not the ones from section 2. Re-run section 2 if
> you want those back.

In [6]:
hits = index.search(
    'T2 fault attribution',
    k=10,
    ext='.md',             # this vault indexes .md and .json only
    path='PhD/**',         # glob, relative to the vault root
    # source='my-wiki',    # the one configured source
    # since='2026-07-01',  # only notes modified since
)

for hit in hits:
    print(f"{hit['matched_by']:<8} {hit['score']:>8.4f}  {hit['citation']}")

rerank     0.8806  PhD/pca-t2-spe-attribution-methods.md:37-57 — PCA T² / SPE Attribution — Two Base Methods > Method 1 — PCA Loading Attribution (T²)
rerank     0.8738  PhD/hawkins-decomposition-t2-fault-diagnosis.md:88-102 — Hawkins' Whitened T² Decomposition for Fault Diagnosis > From Raw Measurements to Contributions — Step by Step
rerank     0.8349  PhD/pca-t2-spe-attribution-methods.md:81-134 — PCA T² / SPE Attribution — Two Base Methods > Worked Numerical Example (4 variables, 2 components)
rerank     0.8204  PhD/pca-t2-spe-attribution-methods.md:135-142 — PCA T² / SPE Attribution — Two Base Methods > Why the Two Methods Give Different Answers
rerank     0.7985  PhD/pca-t2-spe-attribution-methods.md:11-36 — PCA T² / SPE Attribution — Two Base Methods > Setup and Symbols
rerank     0.7400  PhD/pca-t2-spe-attribution-methods.md:58-80 — PCA T² / SPE Attribution — Two Base Methods > Method 2 — PCA Residual Attribution (SPE)
rerank     0.7255  PhD/pca-t2-spe-attribution-methods.md:3-

## 5. As a table
### `PURE` — instant

The same hits laid out for scanning many at once. Plain Python over `hits`; nothing is searched or
loaded.

Uses pandas when it is installed and falls back to printed lines when it is not — pandas is not
part of the `.rag` environment, so the fallback is the normal path.

In [7]:
# pandas is not part of the .rag environment; fall back to plain output if absent.
rows = [
    {'score': h['score'], 'matched_by': h['matched_by'],
     'citation': h['citation'], 'chars': len(h['text'])}
    for h in hits
]
try:
    import pandas as pd
    frame = pd.DataFrame(rows)
    display(frame)
except ModuleNotFoundError:
    for r in rows:
        print(f"{r['score']:>8.4f}  {r['matched_by']:<8} {r['chars']:>5}  {r['citation']}")

  0.8806  rerank     838  PhD/pca-t2-spe-attribution-methods.md:37-57 — PCA T² / SPE Attribution — Two Base Methods > Method 1 — PCA Loading Attribution (T²)
  0.8738  rerank     671  PhD/hawkins-decomposition-t2-fault-diagnosis.md:88-102 — Hawkins' Whitened T² Decomposition for Fault Diagnosis > From Raw Measurements to Contributions — Step by Step
  0.8349  rerank    1354  PhD/pca-t2-spe-attribution-methods.md:81-134 — PCA T² / SPE Attribution — Two Base Methods > Worked Numerical Example (4 variables, 2 components)
  0.8204  rerank     712  PhD/pca-t2-spe-attribution-methods.md:135-142 — PCA T² / SPE Attribution — Two Base Methods > Why the Two Methods Give Different Answers
  0.7985  rerank    1735  PhD/pca-t2-spe-attribution-methods.md:11-36 — PCA T² / SPE Attribution — Two Base Methods > Setup and Symbols
  0.7400  rerank     946  PhD/pca-t2-spe-attribution-methods.md:58-80 — PCA T² / SPE Attribution — Two Base Methods > Method 2 — PCA Residual Attribution (SPE)
  0.7255  rerank 

## 6. Build a prompt block
### `EMBEDS QUERY` · `READS INDEX`

Runs its own search, then formats the passages with their citations already attached — ready to
paste into a conversation with an AI.

This is the point of the whole tool: **it retrieves, something else writes the answer.** Because
every passage arrives with a citation, each claim in that answer can be traced back to a real
location in your notes.

In [8]:
print(index.context_block('Geeignetheitserklärung', k=6, max_chars=1200))

# Retrieved for: Geeignetheitserklärung

## [1] Banking/mifid-wphg-banking-notes.md:107-136 — Geeignetheitserklärung (GEE)  (score 0.952476)
# Geeignetheitserklärung (GEE)

**The GEE explains why the bank's recommendation is suitable for the
customer.**

Created after the suitability assessment.

Contains: - Recommended product(s) - Investment objectives - Risk
profile - Financial situation - Knowledge & experience - Investment
horizon - Explanation of suitability - Warnings - Advisor information

Typical architecture:

``` text
Customer Profile
      │
Risk Profile
      │
Knowledge & Experience
      │
Target Market
      │
Recommendation Engine
      │
      ▼
GEE Generator
      ▼
PDF
```

## [2] Banking/COBA/AVD/knowledge.md:15-28 — Business Knowledge Extraction Report > 3. Key Business Concepts & Glossary  (score 0.905963)
## 3. Key Business Concepts & Glossary
| Term | Business Meaning |
|------|------------------|
| **AVD / FRÜHSTART** | Early Retirement Savings Account (minors

## 7. Inspect one chunk in full
### `PURE` — instant

The single best hit, complete. `anchor` is the raw citation data — a line range here; a page
number, sheet and rows, or notebook cell for other file types.

Reads `hits`, so it shows whatever the **most recent** search cell produced.

In [9]:
if hits:
    top = hits[0]
    print('path        :', top['path'])
    print('heading path:', top['heading_path'])
    print('anchor      :', top['anchor'])
    print('-' * 70)
    print(top['text'])

path        : /Users/Khaled.Alabsi/Library/Mobile Documents/iCloud~md~obsidian/Documents/my-wiki/PhD/pca-t2-spe-attribution-methods.md
heading path: PCA T² / SPE Attribution — Two Base Methods > Method 1 — PCA Loading Attribution (T²)
anchor      : {'line_start': 37, 'line_end': 57}
----------------------------------------------------------------------
## Method 1 — PCA Loading Attribution (T²)

**What it measures:** how much each variable drove the sample too far along directions the model already knows about.

**Step 1 — scores.** Project $x$ into the $c$-dim space:

$$t_k = \sum_{j=1}^{p} P_{j,k} \, x_j, \qquad k = 1,...,c$$

**Step 2 — standardize.**

$$z_k = \frac{t_k}{\sigma_k}$$

**Step 3 — attribute back to variables.**

$$\text{contrib}_j = \sum_{k=1}^{c} |z_k| \cdot |P_{j,k}|$$

Each term multiplies "how abnormal is component $k$" ($|z_k|$) by "how much does variable $j$ feed component $k$" ($|P_{j,k}|$), summed over all $c$ components. Variables with large loadings on the mo

## 8. What did the search actually do?
### `EMBEDS QUERY` · `READS INDEX`

The counts behind a single search:

- **dense hits** — found by meaning (the embedding model)
- **text hits** — found by literal keyword match. **Zero here means keyword search is broken**, and
  exact terms — a name, an ID, `Geeignetheitserklärung` — will retrieve badly even though ordinary
  questions still work.
- **after fuse** — the two lists merged into one ranking
- **reranked** — whether a second model re-scored the top candidates by reading each one against
  the query

This is the first thing to look at when results are worse than you expect.

In [10]:
report = index.search_report('how do I set a boundary when scope creeps mid-task', k=5)

print(f"dense hits : {report.dense_count}")
print(f"text hits  : {report.text_count}")
print(f"after fuse : {report.fused_count}")
print(f"reranked   : {report.reranked}")
for note in report.notes:
    print(f"note       : {note}")

dense hits : 60
text hits  : 60
after fuse : 107
reranked   : True


## 9. Close
### releases RAM · `WRITES` nothing

Closes the store and lets the models go. Nothing on disk is modified.

`Index` is also a context manager, if you prefer `with Index.find() as index:`.

In [11]:
index.close()

---

## What is deliberately *not* in this notebook

**Updating the index is a terminal command, not a cell.** It re-reads your files and re-embeds
whatever changed — heavy, long-running, and it would fight with the models this kernel is already
holding.

```bash
rag update          # after adding or editing notes — only the changed ones are re-read
rag status          # confirm it moved
rag doctor          # when something is wrong
rag index --full    # rebuild from scratch — only needed after a model or chunking change
```

If a note you just wrote does not come back from a search, it is almost always because `rag update`
has not run yet.